# Flight Data Platform - Transform (Camada Silver)

Este notebook lê os dados da **camada bronze** (tabelas Delta no catálogo `workspace.default`) e prepara DataFrames para futuras transformações da camada silver.

## Tabelas bronze lidas

| Tabela | Descrição |
| --- | --- |
| `workspace.default.bronze_opensky_states` | Estado atual das aeronaves (OpenSky `/states/all`) |
| `workspace.default.dim_airports` | Dimensão de aeroportos (OurAirports `airports.csv`) |
| `workspace.default.bronze_opensky_departures` | Partidas do SBGR (OpenSky `/flights/departure`) |

## Imports

Importa as funções do Spark SQL (`pyspark.sql.functions`) e o módulo `builtins` do Python. O `builtins` é necessário porque o wildcard import sobrescreve funções nativas como `round`.

In [0]:
# Imports globais: funções do Spark SQL e builtins do Python
# (builtins é necessário pois 'from pyspark.sql.functions import *' sobrescreve o round nativo)
from pyspark.sql.functions import *
import builtins

## 1. Leitura das Tabelas Bronze

Carrega os DataFrames diretamente das tabelas Delta no catálogo `workspace.default` usando `spark.table()`. Cada tabela é lida em uma célula separada para facilitar o debugging individual.

In [0]:
# Ler tabela bronze: estados das aeronaves (OpenSky states/all)
df_states = spark.table("workspace.default.bronze_opensky_states")
print(f"bronze_opensky_states: {df_states.count()} linhas")
display(df_states.limit(10))

In [0]:
# Ler tabela dimensão: aeroportos (OurAirports airports.csv)
df_airports = spark.table("workspace.default.dim_airports")
print(f"dim_airports: {df_airports.count()} linhas")
display(df_airports.limit(10))

In [0]:
# Ler tabela bronze: partidas do SBGR (OpenSky flights/departure)
df_departures = spark.table("workspace.default.bronze_opensky_departures")
print(f"bronze_opensky_departures: {df_departures.count()} linhas")
display(df_departures.limit(10))

## 2. Análise de Nulos

A função `null_report` percorre todas as colunas de cada DataFrame, conta os valores nulos e calcula a porcentagem. O resultado é uma tabela mostrando apenas as colunas que contêm ao menos 1 valor nulo, ordenada por porcentagem decrescente.

In [0]:
def null_report(df, table_name):
    """Conta nulos por coluna e retorna apenas colunas que têm ao menos 1 valor nulo."""
    total = df.count()
    rows = []
    cols = df.columns  # cacheia schema para evitar Analyze RPC no loop
    for c in cols:
        nulls = df.select(count(when(col(c).isNull(), c)).alias("nulls")).collect()[0]["nulls"]
        if nulls > 0:
            pct = (nulls / total) * 100
            rows.append((table_name, c, nulls, builtins.round(pct, 2)))
    return rows

all_rows = []
for df_obj, name in [(df_states, "bronze_opensky_states"), (df_airports, "dim_airports"), (df_departures, "bronze_opensky_departures")]:
    all_rows.extend(null_report(df_obj, name))

if all_rows:
    spark.createDataFrame(all_rows, ["tabela", "coluna", "nulos", "%_nulos"]).orderBy("tabela", "%_nulos", ascending=False).display()
else:
    print("Nenhuma coluna nula encontrada!")

## 3. Limpeza - Remoção de Colunas 100% Nulas

A função `drop_all_null_columns` percorre cada coluna do DataFrame, conta os nulos, e remove aquelas onde **todas as linhas** são nulas. Isso garante que apenas colunas com dados reais permaneçam na camada silver.

In [0]:
def drop_all_null_columns(df, table_name):
    """Detecta e remove colunas onde TODAS as linhas são nulas (100%)."""
    total = df.count()
    cols = df.columns  # cacheia schema para evitar Analyze RPC no loop
    cols_to_drop = []
    for c in cols:
        nulls = df.select(count(when(col(c).isNull(), c)).alias("nulls")).collect()[0]["nulls"]
        if nulls == total:
            cols_to_drop.append(c)
    if cols_to_drop:
        print(f"[{table_name}] Removendo {len(cols_to_drop)} coluna(s) 100% nulas: {cols_to_drop}")
        return df.drop(*cols_to_drop)
    else:
        print(f"[{table_name}] Nenhuma coluna 100% nula.")
        return df

df_states = drop_all_null_columns(df_states, "bronze_opensky_states")
df_airports = drop_all_null_columns(df_airports, "dim_airports")
df_departures = drop_all_null_columns(df_departures, "bronze_opensky_departures")

## 4. Visualização dos DataFrames Limpos

Exibe as primeiras 10 linhas de cada DataFrame após a remoção das colunas 100% nulas.

In [0]:
# Exibe os 3 DataFrames após a remoção de colunas 100% nulas
print("=== bronze_opensky_states ===")
display(df_states.limit(10))
print("=== dim_airports ===")
display(df_airports.limit(10))
print("=== bronze_opensky_departures ===")
display(df_departures.limit(10))

   
## 5. Transformações Silver - Limpeza e Enriquecimento

Aplica as transformações que preparam os dados da bronze para consumo analítico:

- **Trim de strings**: remove espaços à direita em `callsign` (states e departures)
- **Conversão de timestamps**: `time_position` e `last_contact` (Unix epoch) → `timestamp` legível
- **Desduplicação**: mantém apenas o registro mais recente por `icao24` em states
- **Colunas derivadas**: altitude em pés, velocidade em km/h, flag de voo, categoria de heading
- **Flags de qualidade**: `is_position_complete` indica se lat/long/altitude estão preenchidos
- **Filtragem de dim_airports**: mantém apenas aeroportos com serviço regular
- **Enriquecimento**: join de departures com dim_airports para trazer nome/cidade/país do aeroporto

In [0]:
# --- Transformações: silver_opensky_states ---
from pyspark.sql.window import Window

silver_states = (
    df_states
    # 1. Trim de strings
    .withColumn("callsign", trim(col("callsign")))
    # 2. Converter timestamps Unix epoch -> timestamp legível
    .withColumn("time_position", timestamp_seconds(col("time_position").cast("long")))
    .withColumn("last_contact", timestamp_seconds(col("last_contact").cast("long")))
    # 3. Desduplicação: manter apenas o registro mais recente por icao24
    #    (usa row_number ordenado por last_contact desc)
)

w = Window.partitionBy("icao24").orderBy(col("last_contact").desc())
silver_states = (
    silver_states
    .withColumn("_rn", row_number().over(w))
    .filter(col("_rn") == 1)
    .drop("_rn")
    # 4. Colunas derivadas
    .withColumn("altitude_ft", round(col("baro_altitude") * 3.28084, 2))
    .withColumn("velocity_kmh", round(col("velocity") * 3.6, 2))
    .withColumn("is_airborne", ~col("on_ground"))
    .withColumn("heading_category",
        when(col("true_track").isNull(), None)
        .when((col("true_track") >= 337.5) | (col("true_track") < 22.5), "N")
        .when(col("true_track") < 67.5, "NE")
        .when(col("true_track") < 112.5, "E")
        .when(col("true_track") < 157.5, "SE")
        .when(col("true_track") < 202.5, "S")
        .when(col("true_track") < 247.5, "SW")
        .when(col("true_track") < 292.5, "W")
        .otherwise("NW")
    )
    # 5. Flag de qualidade: posição completa (lat + long + altitude)
    .withColumn("is_position_complete",
        col("latitude").isNotNull()
        & col("longitude").isNotNull()
        & col("geo_altitude").isNotNull()
    )
)

print(f"silver_states: {silver_states.count()} linhas (após dedup)")
display(silver_states.limit(10))

In [0]:
# --- Transformações: silver_dim_airports ---
# Filtrar apenas aeroportos com serviço regular ou de grande/médio porte
# (remove heliports e pequenos aeródromos sem voo comercial)
silver_airports = (
    df_airports
    .filter(
        (col("scheduled_service") == "yes")
        | col("type").isin("large_airport", "medium_airport")
    )
    # Trim de strings
    .withColumn("ident", trim(col("ident")))
    .withColumn("name", trim(col("name")))
    .withColumn("municipality", trim(col("municipality")))
    .withColumn("icao_code", trim(col("icao_code")))
    .withColumn("iata_code", trim(col("iata_code")))
)

print(f"silver_airports: {silver_airports.count()} linhas (após filtro)")
display(silver_airports.limit(10))

In [0]:
# --- Transformações: silver_opensky_departures ---
# 1. Trim de strings
silver_departures = df_departures.withColumn("callsign", trim(col("callsign")))

# 2. Enriquecimento: join com silver_airports para trazer nome, cidade e país
silver_departures = (
    silver_departures.alias("d")
    .join(
        silver_airports.alias("a"),
        col("d.query_airport") == col("a.icao_code"),
        "left"
    )
    .select(
        col("d.icao24"),
        col("d.callsign"),
        col("d.query_airport").alias("departure_icao"),
        col("a.name").alias("departure_airport_name"),
        col("a.municipality").alias("departure_city"),
        col("a.iso_country").alias("departure_country"),
        col("a.elevation_ft").alias("departure_elevation_ft"),
        col("a.type").alias("departure_airport_type"),
        col("d.ingestion_ts"),
    )
)

print(f"silver_departures: {silver_departures.count()} linhas (após enriquecimento)")
display(silver_departures.limit(14))

   
## 6. Persistência em Delta (Camada Silver)

Salva os DataFrames transformados como tabelas Delta no catálogo `workspace.default` com overwrite + `overwriteSchema=true`. As tabelas silver estão prontas para consumo analítico e construção da camada gold.

In [0]:
# Persistir as tabelas silver em Delta
silver_tables = {
    "workspace.default.silver_opensky_states": silver_states,
    "workspace.default.silver_dim_airports": silver_airports,
    "workspace.default.silver_opensky_departures": silver_departures,
}

for table_name, df in silver_tables.items():
    print(f"Salvando {table_name} ({df.count()} linhas)...")
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    print(f"  ✓ {table_name} salva com sucesso!")

print("\nTodas as tabelas silver foram persistidas!")